# Notebook 14 — RAG Pipeline Deep Dive

**What you'll learn:**
- End-to-end file ingestion: markdown and PDF
- SHA-256 deduplication with the SQLite file registry
- Hybrid PDF conversion (pymupdf4llm + LLM vision)
- Page complexity classification
- Building searchable knowledge from raw files

**Prerequisites:** Notebook 13

**Time:** ~35 minutes

## 1. Introduction: The Full RAG Pipeline

When Lang-PINN searches for relevant PINN techniques, it doesn't query the internet — it searches a local **knowledge store** built from your own documents: papers, notes, experiment logs.

The pipeline from raw file to searchable knowledge looks like this:

```
raw file (markdown / PDF)
        |
        v
  compute SHA-256 hash
        |
        v
  FileRegistry.lookup(hash)
        |             |
     found          not found
   (skip) ----    -----> convert to markdown
                              |
                              v
                    parse into DocumentTree
                    (hierarchical sections)
                              |
                              v
                    KnowledgeStore.add_document()
                    store.save()  →  JSON on disk
                              |
                              v
                    FileRegistry.register()
                    SQLite: hash → doc_id
                              |
                              v
                    SearchEngine.from_store()
                    BM25 index in memory
                              |
                              v
                    engine.search(query)
                    → ranked doc_ids
```

The three main components:
- **`KnowledgeStore`** — persists indexed `DocumentTree` objects as JSON
- **`FileRegistry`** — SQLite database tracking file hashes to prevent duplicate work
- **`SearchEngine`** — in-memory BM25 index for fast keyword search

The entry point that wires them together is **`ingest_file()`**.

## 2. Setting Up

In [ ]:
import tempfile
from pathlib import Path

from rag import KnowledgeStore, FileRegistry, SearchEngine, ingest_file, IngestResult
from rag.indexing.pdf import classify_pages, get_page_count

# Create a self-contained temp directory for this notebook.
# Nothing written here pollutes the repo.
_tmp = tempfile.mkdtemp(prefix="nb14_rag_")
tmp_path = Path(_tmp)

store_dir = tmp_path / "kb"
db_path   = tmp_path / "kb" / "registry.db"

store    = KnowledgeStore(store_dir)
registry = FileRegistry(db_path)

print(f"Working directory : {tmp_path}")
print(f"Knowledge store   : {store_dir}")
print(f"File registry     : {db_path}")
print(f"Documents indexed : {len(store.list_documents())}")
print(f"Registry entries  : {registry.count()}")

## 3. Ingesting a Markdown File

We'll create a sample markdown document about a fictional PDE — the **Kuramoto-Sivashinsky equation** — and run it through the pipeline.

In [ ]:
md_content = """\
# Kuramoto-Sivashinsky Equation

The Kuramoto-Sivashinsky (KS) equation is a prototypical model for spatiotemporal chaos:

$$u_t + u u_x + u_{xx} + u_{xxxx} = 0$$

It arises in flame-front propagation, falling thin films, and plasma drift waves.

## PINN Considerations

The fourth-order term $u_{xxxx}$ requires autograd to be called twice:

- Compute $u_x$ with `grad(u, x, create_graph=True)`
- Compute $u_{xx}$ from $u_x$
- Compute $u_{xxx}$ from $u_{xx}$
- Compute $u_{xxxx}$ from $u_{xxx}$

Each call must use `create_graph=True` except the last.

## Known Difficulties

- Chaotic dynamics mean the exact solution is not useful as a benchmark past short times.
- Standard PINNs struggle with multi-scale energy cascades.
- Domain decomposition (XPINNs) or Fourier feature networks help.

## Recommended Architecture

- Network: 6 layers, 128 neurons, `tanh` activation
- Fourier feature embedding with frequency range `[1, 40]`
- Collocation: 10 000 points with RAR refinement near steep gradients
- Loss weights: `{physics: 1.0, ic: 20.0, bc: 5.0}`
- Optimizer: Adam 5 000 epochs then L-BFGS
"""

md_path = tmp_path / "kuramoto_sivashinsky.md"
md_path.write_text(md_content)

print(f"Created: {md_path.name}")
print(f"Size   : {md_path.stat().st_size} bytes")

In [ ]:
result = ingest_file(md_path, store=store, registry=registry)

print("IngestResult:")
print(f"  status    : {result.status}")
print(f"  file_path : {Path(result.file_path).name}")
print(f"  file_hash : {result.file_hash[:16]}…")
print(f"  doc_id    : {result.doc_id}")
print(f"  message   : {result.message}")
print()
print(f"Store now holds {len(store.list_documents())} document(s)")
print(f"Registry now holds {registry.count()} entry(s)")

The `IngestResult` is a lightweight dataclass — not a big object:

| Field | Meaning |
|---|---|
| `status` | `"indexed"`, `"skipped"`, or `"error"` |
| `file_hash` | SHA-256 hex digest of the file bytes |
| `doc_id` | Slug used as the key inside `KnowledgeStore` |
| `message` | Human-readable description of what happened |

The status is `"indexed"` on first run. Let's see what happens if we call it again with the same file.

## 4. Deduplication

The registry stores a SHA-256 hash of every ingested file. On the next `ingest_file()` call the hash is recomputed and looked up — if found, the file is skipped. No re-parsing, no re-indexing.

In [ ]:
# Ingest the same file again — content unchanged.
result2 = ingest_file(md_path, store=store, registry=registry)

print(f"status  : {result2.status}")
print(f"message : {result2.message}")
print(f"doc_id  : {result2.doc_id}   (same as before)")
print()
print(f"Store documents: {len(store.list_documents())}  (unchanged)")

Now modify the file — even adding a single character changes the hash:

In [ ]:
# Append a new section to the file.
with md_path.open("a") as f:
    f.write("\n## References\n\n- Kuramoto & Tsuzuki (1975). On the formation of dissipative structures.\n")

old_hash = result.file_hash
new_hash = FileRegistry.compute_hash(md_path)

print(f"Old hash : {old_hash[:16]}…")
print(f"New hash : {new_hash[:16]}…")
print(f"Changed  : {old_hash != new_hash}")

In [ ]:
result3 = ingest_file(md_path, store=store, registry=registry)

print(f"status    : {result3.status}")
print(f"doc_id    : {result3.doc_id}")
print(f"file_hash : {result3.file_hash[:16]}…  (new hash)")
print(f"message   : {result3.message}")
print()
print(f"Store documents: {len(store.list_documents())}")
print(f"Registry entries: {registry.count()}  (both hashes recorded)")

The registry records **both** hashes — the original ingestion and the updated one. The store also upserts the document: the same `doc_id` now reflects the new content.

## 5. Force Re-Index

Sometimes you want to re-process a file even though its content hasn't changed — for example after updating the indexer's parsing logic. Pass `force=True` to override the dedup check.

In [ ]:
# Confirm the current file is already indexed.
print(f"is_indexed: {registry.is_indexed(md_path)}")

result_forced = ingest_file(md_path, store=store, registry=registry, force=True)

print(f"\nWith force=True:")
print(f"  status  : {result_forced.status}")
print(f"  message : {result_forced.message}")

With `force=True` the registry lookup is skipped entirely. The file is re-indexed from scratch and the registry entry is updated in place (same hash, new `updated_at` timestamp).

## 6. Ingesting with Metadata

You can pass a `metadata` dict to enrich the `DocumentTree` before it's stored. This metadata is included in the BM25 index so it boosts search relevance for semantically tagged queries.

In [ ]:
# A second document: notes on the Allen-Cahn equation.
ac_content = """\
# Allen-Cahn Equation

The Allen-Cahn equation models phase separation:

$$u_t = \\epsilon^2 u_{xx} + u - u^3$$

## Boundary Conditions

Periodic boundary conditions are standard for Allen-Cahn benchmarks.

## Challenges

- Sharp interface layers require high collocation density.
- The cubic nonlinearity slows convergence.
"""

ac_path = tmp_path / "allen_cahn.md"
ac_path.write_text(ac_content)

ac_result = ingest_file(
    ac_path,
    store=store,
    registry=registry,
    metadata={
        "pde_type": "reaction-diffusion",
        "techniques": ["phase-field", "fourier-features", "periodic-bc"],
        "keywords": ["Allen-Cahn", "phase separation", "interface", "nonlinear"],
    },
)

print(f"status : {ac_result.status}")
print(f"doc_id : {ac_result.doc_id}")
print()

# Confirm metadata is stored.
meta = store.get_metadata(ac_result.doc_id)
print(f"pde_type   : {meta.pde_type}")
print(f"techniques : {meta.techniques}")
print(f"keywords   : {meta.keywords}")

Supported metadata keys map directly to `DocumentTree` fields:

| Key | Type | Purpose |
|---|---|---|
| `pde_type` | `str` | E.g. `"reaction-diffusion"`, `"hyperbolic"` |
| `techniques` | `list[str]` | Methods used: `["RAR", "fourier-features"]` |
| `keywords` | `list[str]` | Free-form tags for BM25 boosting |
| `known_issues` | `list[str]` | Caveats: `["slow convergence"]` |

## 7. The File Registry

The `FileRegistry` is a thin wrapper around a single SQLite table. Every ingested file gets one row: its hash, path, size, page count (for PDFs), and which converter was used.

In [ ]:
print(f"Total entries : {registry.count()}")
print()

records = registry.list_all()
for rec in records:
    print(f"  file_name  : {rec.file_name}")
    print(f"  doc_id     : {rec.doc_id}")
    print(f"  converter  : {rec.converter}")
    print(f"  file_hash  : {rec.file_hash[:16]}…")
    print(f"  created_at : {rec.created_at[:19]}")
    print()

In [ ]:
# is_indexed() recomputes the hash on disk — not a path-based lookup.
print(f"is_indexed(allen_cahn.md)       : {registry.is_indexed(ac_path)}")
print(f"is_indexed(non_existent_file)   : ", end="")
non_existing = tmp_path / "missing.md"
non_existing.write_text("temp")          # create so hash can be computed
print(registry.is_indexed(non_existing)) # False — not yet registered
non_existing.unlink()

# lookup_by_path() finds the most recent record for a given file path.
rec = registry.lookup_by_path(ac_path)
print(f"\nlookup_by_path result:")
print(f"  file_name : {rec.file_name}")
print(f"  doc_id    : {rec.doc_id}")
print(f"  converter : {rec.converter}")

### Why SQLite? Why not just scan the store directory?

The registry solves a subtlety: the **same document content** can appear at different paths (e.g. a paper you downloaded twice with different filenames). Without content-based dedup, you'd index it twice and waste storage and search quality.

SQLite was chosen over a plain JSON file because:
- Atomic writes prevent corruption on crash
- Indexed columns (`doc_id`, `file_path`) make lookups O(log n)
- The schema is intentionally minimal — straightforward to migrate to PostgreSQL by replacing the connection string with `psycopg2` and adjusting the `CREATE TABLE` syntax.

## 8. Hybrid PDF Conversion

PDFs are not all created equal. A methods section is dense text — `pymupdf4llm` handles it perfectly and costs nothing. A figure-heavy page full of flowcharts and architecture diagrams produces nearly empty text extraction — that's where LLM vision earns its cost.

The **2-tier hybrid approach**:

```
PDF
 ├── classify_pages()
 │       → "simple"  (text_tokens >= 30 AND image_area <= 40%)
 │       → "complex" (text_tokens <  30 OR  image_area >  40%)
 │
 ├── simple pages  → pymupdf4llm  (free, fast, batch)
 └── complex pages → LLM vision   (paid, async, max 3 concurrent)
```

Complexity thresholds (from `rag/indexing/pdf.py`):
- `_MIN_TEXT_TOKENS = 30` — fewer than 30 words → complex
- `_IMAGE_AREA_THRESHOLD = 0.4` — images cover > 40% of page area → complex

Let's create a simple test PDF and run `classify_pages()` on it.

In [ ]:
import pymupdf

pdf_path = tmp_path / "test_pde_notes.pdf"

doc = pymupdf.open()

# Page 0: text-heavy — should classify as "simple"
page0 = doc.new_page(width=595, height=842)  # A4
text_lines = [
    "Burgers Equation: u_t + u*u_x = nu*u_xx",
    "",
    "The Burgers equation is a fundamental PDE combining nonlinear convection",
    "with diffusion. It appears in gas dynamics, traffic flow, and acoustics.",
    "",
    "PINN approach:",
    "  1. Sample N collocation points in the domain.",
    "  2. Compute u_t and u_x via autograd.",
    "  3. Penalise the physics residual u_t + u*u_x - nu*u_xx.",
    "  4. Enforce initial condition u(x, 0) = -sin(pi*x).",
    "  5. Enforce Dirichlet BCs u(-1, t) = u(1, t) = 0.",
    "",
    "With nu = 0.01/pi, a sharp shock forms near t = 0.4.",
    "Use 10 000 collocation points concentrated near the shock.",
]
y = 72.0
for line in text_lines:
    page0.insert_text((72, y), line, fontsize=11)
    y += 16

# Page 1: mostly blank — should classify as "complex" (low text tokens)
page1 = doc.new_page(width=595, height=842)
page1.insert_text((72, 420), "Figure 1: Network architecture diagram", fontsize=10)

doc.save(str(pdf_path))
doc.close()

print(f"Created PDF: {pdf_path.name}")
print(f"Pages      : {get_page_count(pdf_path)}")

In [ ]:
classifications = classify_pages(pdf_path)

print("Page classification results:")
print(f"{'Page':>5}  {'Complexity':12}  {'Text tokens':>11}  {'Image area':>10}")
print("-" * 48)
for c in classifications:
    print(
        f"  {c['page_num']:>3}  {c['complexity']:12}  "
        f"{c['text_tokens']:>11}  {c['image_area_ratio']:>10.3f}"
    )

Page 0 has enough text to be `"simple"` — `pymupdf4llm` handles it for free.  
Page 1 has almost no text — classified `"complex"`, so in full hybrid mode it would be sent to LLM vision.

### Enabling hybrid mode in `ingest_file()`

```python
from llm_provider import LLMClient

llm = LLMClient()  # configured via ANTHROPIC_API_KEY

result = ingest_file(
    "complex_paper.pdf",
    store=store,
    registry=registry,
    hybrid_pdf=True,     # enable two-tier conversion
    llm_client=llm,      # required for complex-page vision calls
)
```

If `hybrid_pdf=True` but `llm_client=None`, the pipeline falls back to `pymupdf4llm` for all pages and logs a warning. The registry records which pages used LLM (`llm_pages` column) for cost auditing.

In [ ]:
# Ingest the PDF without an LLM client (pymupdf4llm for all pages).
pdf_result = ingest_file(pdf_path, store=store, registry=registry)

print(f"status    : {pdf_result.status}")
print(f"doc_id    : {pdf_result.doc_id}")
print(f"message   : {pdf_result.message}")
print()

# Inspect the registry record for cost tracking.
rec = registry.lookup_by_path(pdf_path)
print(f"converter : {rec.converter}")
print(f"page_count: {rec.page_count}")
print(f"llm_pages : '{rec.llm_pages}'  (empty = all pages used pymupdf4llm)")

## 9. Searching Ingested Documents

`SearchEngine.from_store()` builds an in-memory BM25 index from everything in the store. It concatenates each document's metadata fields (pde_type, techniques, keywords) and all node titles, summaries, and text — giving a single searchable string per document.

In [ ]:
engine = SearchEngine.from_store(store)

# Show what's in the store.
print("Documents in store:")
for meta in store.list_documents():
    print(f"  {meta.doc_id:40s}  nodes={meta.node_count}  tokens={meta.total_tokens}")

In [ ]:
queries = [
    "fourth order chaos",
    "phase separation interface nonlinear",
    "sharp gradients shock burgers",
    "collocation points fourier features",
]

for q in queries:
    hits = engine.search(q, top_k=3)
    print(f"Query: '{q}'")
    if hits:
        for rank, doc_id in enumerate(hits, 1):
            print(f"  {rank}. {doc_id}")
    else:
        print("  (no results)")
    print()

BM25 is a bag-of-words model — it works well when your queries share vocabulary with the document. The metadata fields (`pde_type`, `techniques`, `keywords`) are especially valuable here because they let you tag documents with controlled vocabulary that users are likely to search for.

For richer semantic matching, `rag.retrieve()` layers an LLM reasoning step on top of BM25 results — but that requires an `LLMClient` and is covered separately.

## 10. Full Pipeline Summary

```
ingest_file(path, store, registry, ...)
│
├── FileRegistry.compute_hash(path)         # SHA-256
│
├── registry.lookup(hash)                   # dedup check
│       └── found? → return IngestResult(status="skipped")
│
├── MarkdownIndexer.index_file_sync(path)   # parse
│       ├── .md / .txt  → split by headings into TreeNodes
│       └── .pdf        → pdf_to_markdown() or hybrid_pdf_to_markdown()
│                           then split by headings
│
├── enrich tree with metadata dict
│
├── store.add_document(tree)                # persist JSON
├── store.save()                            # write manifest.json
│
├── registry.register(path, ...)            # record in SQLite
│
└── return IngestResult(status="indexed", doc_id=...)

Later:
SearchEngine.from_store(store)              # build BM25 index
engine.search(query, top_k=5)              # ranked doc_ids
store.get_document(doc_id)                  # load DocumentTree
retrieve(store, query, llm_client)          # full RAG retrieval
```

### Key design choices

| Choice | Rationale |
|---|---|
| Content-hash dedup | Filename renames don't cause re-indexing; file moves don't either |
| SQLite registry | Zero dependency, crash-safe, migratable to Postgres |
| Hierarchical tree | Preserves document structure for LLM reasoning over sections |
| BM25 first pass | No embeddings, no vector DB, no network call — fast and free |
| Hybrid PDF | Optimises cost: pay LLM only for pages that truly need it |

## 11. Exercise

Try the pipeline with a real document:

1. Download a PINN paper PDF (e.g. the original Raissi et al. 2019 paper)
2. Ingest it with `metadata={"pde_type": "...", "techniques": [...]}`
3. Run `classify_pages()` on it — how many pages are complex?
4. Build a `SearchEngine` and search for `"neural network training"` and `"Navier-Stokes"`
5. Ingest your own notes markdown file and search across both documents

In [ ]:
# Your turn!
#
# paper_path = Path("path/to/raissi_2019.pdf")
#
# result = ingest_file(
#     paper_path,
#     store=store,
#     registry=registry,
#     metadata={
#         "pde_type": "Navier-Stokes",
#         "techniques": ["PINN", "data-driven"],
#         "keywords": ["Raissi", "physics-informed", "hidden fluid mechanics"],
#     },
# )
# print(result)
#
# pages = classify_pages(paper_path)
# n_complex = sum(1 for p in pages if p["complexity"] == "complex")
# print(f"Complex pages: {n_complex}/{len(pages)}")
#
# engine = SearchEngine.from_store(store)
# print(engine.search("Navier-Stokes velocity field"))

In [ ]:
# Clean up the temp directory when you're done experimenting.
import shutil
registry.close()
shutil.rmtree(tmp_path)
print(f"Cleaned up: {tmp_path}")

## Summary: The Complete PINN Curriculum

| Notebook | Topic | Key Skill |
|----------|-------|----------|
| 01 | What are PINNs? | Motivation, landscape |
| 02 | Automatic differentiation | `torch.autograd.grad` |
| 03 | First PINN from scratch | Raw PyTorch PINN |
| 04 | Data vs Physics vs Hybrid | Loss function design |
| 05 | PDEs and boundary conditions | Multi-term losses |
| 06 | Training tricks | Ansatz, weighting, scheduling |
| 07 | Parametric and inverse | Parameters as inputs |
| 08 | Honest assessment | When (not) to use PINNs |
| 09 | Inverse Navier-Stokes | Advanced: Re inference |
| 10 | Lang-PINN intro | 3 agents, 3 modes |
| 11 | Hybrid mode | Feedback loop, quality scoring |
| 12 | Bring your own PDE | End-to-end workflow |
| 13 | RAG intro | Knowledge store, BM25 search |
| **14** | **RAG pipeline deep dive** | **Ingest, dedup, hybrid PDF, search** |